# تحضير بيانات SFT (Supervised Fine-Tuning)

بما أن داتا `Ahmed-Selem/Shifaa_Arabic_Mental_Health_Consultations` هي داتا عربية ممتازة وجاهزة (تحتوي على سؤال المريض وإجابة الطبيب)، فلا حاجة لنا لاستخدام OpenRouter أو الذكاء الاصطناعي لترجمتها أو تعديلها هنا.

سنقوم في هذا النوتبوك ببساطة بتحميل الداتا من المسار المحلي الذي نزلناها فيه، وتغليفها بتنسيق `ChatML` الذي يتوقعه نموذج `Qwen`، وحفظها لتكون جاهزة للتدريب الفوري.

In [ ]:
!pip install -q datasets pandas tqdm

In [ ]:
import json
import os
from datasets import load_from_disk
from tqdm.notebook import tqdm

# تحميل الداتا العربية (Shifaa) من المسار المحلي الخاص بك
print("Loading dataset from local disk...")
dataset_path = "../../local_datasets/Ahmed-Selem_Shifaa_Arabic_Mental_Health_Consultations"
dataset = load_from_disk(dataset_path)

# يمكنك استخدام dataset كاملة (35 ألف ريكورد)، ولكن للسرعة التجريبية سنأخذ أول 1000
sample_data = dataset.select(range(1000))
print(f"Loaded {len(sample_data)} rows for processing.")

In [ ]:
def format_to_chatml(question, answer):
    return {
        "messages": [
            {"role": "system", "content": "أنت طبيب نفسي عربي خبير. تجيب على استشارات المرضى بتعاطف، واحترافية، ودقة طبية عالية."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]
    }

In [ ]:
chatml_dataset = []

print("Formatting to ChatML...")
for row in tqdm(sample_data):
    q = row['Question']
    a = row['Answer']
    
    # التأكد من أن السؤال والإجابة ليسا فارغين
    if q and a and isinstance(q, str) and isinstance(a, str):
        chatml_dataset.append(format_to_chatml(q, a))

# حفظ النتيجة في مجلد processed_data
os.makedirs("processed_data", exist_ok=True)
output_path = "processed_data/chatml_dataset.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chatml_dataset, f, ensure_ascii=False, indent=2)

print(f"تم بنجاح تحويل وحفظ {len(chatml_dataset)} استشارة نفسية بتنسيق ChatML في {output_path}")